# StreamGuard Phase 4: Full Model Training & Ablation Study (GPU)

**Purpose**: Run the 6-configuration ablation study on Colab GPU for the conference paper.

**Why Colab?** Full training across 6 configs requires 18-30 GPU hours. Colab Pro with A100/V100 makes this feasible.

**Prerequisites**:
1. Phase 3 outputs uploaded to Google Drive (`StreamGuard/` folder)
2. Runtime set to **GPU** (T4 minimum, A100 recommended)
3. Colab Pro recommended for long training runs

---

## Data Requirements

Upload **one zip file** to `My Drive/StreamGuard/` before starting:

| File | Size | Source | Purpose |
|------|------|--------|---------|
| `streamguard_data.zip` | ~2-3 GB | See below | All 4 data files compressed |

The zip should contain these 4 files (flat, no subdirectories):

| File | Uncompressed Size | Source | Purpose |
|------|-------------------|--------|---------|
| `train.h5` | ~3.2 GB | Stage 7 split output | Training graphs (42,492 samples) |
| `val.h5` | ~400 MB | Stage 7 split output | Validation graphs (5,311 samples) |
| `test.h5` | ~400 MB | Stage 7 split output | Test graphs (5,312 samples) |
| `samples.jsonl` | ~200 MB | Stage 3 CFA output | Code text for CodeBERT tokenization |

All 6 configs use the **same** data files (mandatory invariant R-05).

> **Tip**: HDF5 and JSONL compress well. The zip will be ~50-60% of the original size, making upload much faster.

---

## Ablation Configs (Paper Table 2)

| Config | Description | Key Flags | Expected F1 |
|--------|-------------|-----------|-------------|
| **A** | CodeBERT sequence only | use_graph=False, lambda_cfa=0 | ~0.79 |
| **B** | + type-blind GGNN, 3-CPG | type_aware=False, lambda_cfa=0 | ~0.83 |
| **B'** | + type-aware GGNN, 3-CPG | type_aware=True, lambda_cfa=0 | ~0.85 |
| **C** | B' + CFA contrastive | lambda_cfa=0.5 | ~0.89 |
| **D** | C + TPG (4-CPG) | cpg=[AST,CFG,DFG,TPG] | ~0.91 |
| **E** | D + inter-proc (Full) | use_interproc=True | ~0.93 |

---

## Risk Mitigations Baked In

This notebook implements pre-flight checks for all 8 CRITICAL risks and 12 HIGH risks from the Phase 4 Risk Analysis document. See Cell 5 for the full check list.

## Cell 1: Verify GPU Runtime

In [ ]:
import torch
import sys
import os

if not torch.cuda.is_available():
    raise RuntimeError(
        "NO GPU DETECTED!\n"
        "Go to: Runtime > Change runtime type > Hardware accelerator > GPU (T4)\n"
        "Then restart the runtime and re-run this cell."
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1024**3
print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
print(f"CUDA: {torch.version.cuda}")
print(f"PyTorch: {torch.__version__}")

# VRAM guidance
if gpu_mem < 14:
    print(f"\nWARN: {gpu_mem:.0f} GB VRAM — use batch_size=4 and gradient_accumulation=8")
    print("Recommended: Colab Pro with A100 (40 GB) for full training")
elif gpu_mem < 40:
    print(f"\nT4/V100 detected — batch_size=8, gradient_accumulation=4 (default)")
else:
    print(f"\nA100 detected — batch_size=16, gradient_accumulation=2 for faster training")

# Quick CUDA test
x = torch.randn(100, 824, device='cuda')
print(f"\nCUDA tensor test: OK ({x.shape})")

## Cell 2: Mount Google Drive

Your Drive should contain:
```
My Drive/StreamGuard/
  streamguard_data.zip   (contains train.h5, val.h5, test.h5, samples.jsonl)
```

**How to prepare** (run on your local machine):

**PowerShell (Windows):**
```powershell
Compress-Archive -Path "training\data\final\train.h5", "training\data\final\val.h5", "training\data\final\test.h5", "training\data\processed\with_cfa\samples.jsonl" -DestinationPath "streamguard_data.zip"
```

**Linux/Mac:**
```bash
zip streamguard_data.zip \
    training/data/final/train.h5 \
    training/data/final/val.h5 \
    training/data/final/test.h5 \
    training/data/processed/with_cfa/samples.jsonl
```

Then upload `streamguard_data.zip` to `My Drive/StreamGuard/`.

> **Note**: You can also upload the 4 files individually (without zipping) — the notebook auto-detects both formats.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ---- Configure paths ----
DRIVE_FOLDER = '/content/drive/MyDrive/StreamGuard'

# Local working directories (Colab fast SSD)
WORK_DIR = '/content/streamguard'
DATA_DIR = f'{WORK_DIR}/training/data/final'
CODE_JSONL_DIR = f'{WORK_DIR}/training/data/processed/with_cfa'
CHECKPOINT_DIR = f'{WORK_DIR}/training/checkpoints'
RESULTS_DIR = f'{WORK_DIR}/results'

# Verify Drive folder exists
if not os.path.exists(DRIVE_FOLDER):
    raise FileNotFoundError(
        f"Drive folder not found: {DRIVE_FOLDER}\n"
        f"Create 'StreamGuard' folder in My Drive and upload data files."
    )

# List available files
print(f"Drive folder: {DRIVE_FOLDER}")
for f in sorted(os.listdir(DRIVE_FOLDER)):
    fpath = os.path.join(DRIVE_FOLDER, f)
    if os.path.isfile(fpath):
        size_mb = os.path.getsize(fpath) / 1024 / 1024
        print(f"  {f}: {size_mb:.1f} MB")

## Cell 3: Clone Repo + Install Dependencies

In [ ]:
%%bash
# Clone repo (or pull latest)
if [ ! -d "/content/streamguard" ]; then
    git clone https://github.com/VimalSajanGeorge/streamguard.git /content/streamguard
else
    cd /content/streamguard && git pull
fi

In [ ]:
# Install dependencies for Phase 4 training
!pip install -q \
    transformers==4.44.0 \
    torch-geometric \
    torch-scatter torch-sparse \
    mlflow \
    loguru \
    h5py \
    scikit-learn \
    numpy \
    'datasketch>=1.6.0,<1.7.0'

# Verify critical imports
import transformers, torch, torch_geometric, h5py, mlflow, sklearn
from loguru import logger
print(f"transformers: {transformers.__version__}")
print(f"torch: {torch.__version__} (CUDA: {torch.cuda.is_available()})")
print(f"torch_geometric: {torch_geometric.__version__}")
print(f"h5py: {h5py.__version__}")
print(f"mlflow: {mlflow.__version__}")
print(f"sklearn: {sklearn.__version__}")

## Cell 4: Extract & Copy Data to Local SSD

If a `streamguard_data.zip` is found on Drive, it is extracted first. Individual files are also supported.

Training reads HDF5 files thousands of times per epoch. Copying to Colab's local SSD is much faster than reading from Drive.

In [ ]:
import shutil
import time
import zipfile

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CODE_JSONL_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── Step 1: Extract zip if present ──────────────────────────────────
ZIP_PATH = os.path.join(DRIVE_FOLDER, 'streamguard_data.zip')

if os.path.exists(ZIP_PATH):
    zip_size_mb = os.path.getsize(ZIP_PATH) / 1024**2
    print(f"Found streamguard_data.zip ({zip_size_mb:.0f} MB). Extracting...")

    extract_dir = os.path.join(DRIVE_FOLDER, '_extracted')
    os.makedirs(extract_dir, exist_ok=True)

    start = time.time()
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        names = zf.namelist()
        print(f"  Zip contains {len(names)} entries: {names}")
        zf.extractall(extract_dir)
    elapsed = time.time() - start
    print(f"  Extracted in {elapsed:.0f}s")

    # Find each required file — handles both flat zips and zips with directory paths
    # (e.g., PowerShell Compress-Archive preserves full paths like training\data\final\train.h5)
    for fname in ['train.h5', 'val.h5', 'test.h5', 'samples.jsonl']:
        target = os.path.join(DRIVE_FOLDER, fname)
        if os.path.exists(target):
            print(f"  {fname}: already exists on Drive, skipping.")
            continue

        # Search extracted tree for the file by basename
        found = None
        for root, dirs, files in os.walk(extract_dir):
            if fname in files:
                found = os.path.join(root, fname)
                break

        if found:
            shutil.move(found, target)
            print(f"  Moved {fname} to {DRIVE_FOLDER}")
        else:
            print(f"  WARNING: {fname} not found in zip!")

    # Clean up extracted tree
    shutil.rmtree(extract_dir, ignore_errors=True)
    print("  Zip extraction complete.\n")
else:
    print("No zip found — expecting individual files on Drive.\n")

# ── Step 2: Copy files from Drive to local SSD ─────────────────────
REQUIRED_FILES = {
    'train.h5': DATA_DIR,
    'val.h5': DATA_DIR,
    'test.h5': DATA_DIR,
    'samples.jsonl': CODE_JSONL_DIR,
}

for fname, dest_dir in REQUIRED_FILES.items():
    src = os.path.join(DRIVE_FOLDER, fname)
    dst = os.path.join(dest_dir, fname)

    if not os.path.exists(src):
        raise FileNotFoundError(
            f"MISSING: {src}\n"
            f"Upload {fname} (or streamguard_data.zip containing it) "
            f"to Google Drive > StreamGuard folder."
        )

    if os.path.exists(dst):
        src_size = os.path.getsize(src)
        dst_size = os.path.getsize(dst)
        if src_size == dst_size:
            print(f"  {fname}: already copied ({dst_size / 1024**2:.0f} MB). Skipping.")
            continue

    print(f"  Copying {fname} ({os.path.getsize(src) / 1024**2:.0f} MB)...", end=' ')
    start = time.time()
    shutil.copy2(src, dst)
    elapsed = time.time() - start
    print(f"done ({elapsed:.0f}s)")

# ── Step 3: Restore checkpoints from previous runs ─────────────────
drive_ckpt_dir = os.path.join(DRIVE_FOLDER, 'checkpoints')
if os.path.exists(drive_ckpt_dir):
    ckpt_files = [f for f in os.listdir(drive_ckpt_dir) if f.endswith('.pt')]
    if ckpt_files:
        print(f"\nRestoring {len(ckpt_files)} checkpoints from Drive...")
        for f in ckpt_files:
            src = os.path.join(drive_ckpt_dir, f)
            dst = os.path.join(CHECKPOINT_DIR, f)
            if not os.path.exists(dst):
                shutil.copy2(src, dst)
                print(f"  Restored: {f}")

print(f"\nData ready on local SSD.")

## Cell 5: Pre-flight Risk Checks

Validates all critical risk mitigations BEFORE any training starts. If any check fails, training must not proceed.

Checks implemented:
- **R-01**: test.h5 path distinct from val.h5
- **R-03**: Model uses GroupNorm (not BatchNorm)
- **R-05/R-06**: All configs share same data paths and seed=42
- **R-15**: Config B has dedicated single_conv (type-blind)
- **R-17**: Edge types in HDF5 are in {0,1,2,3}
- **R-18**: pair_id attributes present in HDF5
- **R-19**: Code lookup JSONL loads successfully
- **R-29**: val.h5 and test.h5 are different files

In [ ]:
import sys
sys.path.insert(0, WORK_DIR)

import h5py
import json
import numpy as np
import torch.nn as nn

TRAIN_H5 = os.path.join(DATA_DIR, 'train.h5')
VAL_H5 = os.path.join(DATA_DIR, 'val.h5')
TEST_H5 = os.path.join(DATA_DIR, 'test.h5')
CODE_JSONL = os.path.join(CODE_JSONL_DIR, 'samples.jsonl')

checks_passed = 0
checks_total = 0

def check(name, condition, detail=""):
    global checks_passed, checks_total
    checks_total += 1
    if condition:
        checks_passed += 1
        print(f"  PASS: {name}")
    else:
        print(f"  FAIL: {name} -- {detail}")

print("=" * 60)
print("PRE-FLIGHT RISK CHECKS")
print("=" * 60)

# ---- R-29: val.h5 != test.h5 ----
check("R-29: val.h5 != test.h5",
      os.path.abspath(VAL_H5) != os.path.abspath(TEST_H5),
      "val.h5 and test.h5 point to same file!")

# ---- HDF5 files exist ----
for name, path in [('train.h5', TRAIN_H5), ('val.h5', VAL_H5), ('test.h5', TEST_H5)]:
    check(f"{name} exists", os.path.exists(path), f"{path} not found")

# ---- R-17: Edge types in {0,1,2,3} ----
# ---- R-18: pair_id attrs present ----
for name, path in [('train.h5', TRAIN_H5), ('val.h5', VAL_H5)]:
    if os.path.exists(path):
        with h5py.File(path, 'r') as f:
            if 'metadata' in f and 'graphs' in f:
                # Layout A
                n_samples = len(f['metadata']['labels'])
                check(f"{name}: has samples", n_samples > 0, "No samples")

                # Check pair_ids exist (R-18)
                has_pairs = 'pair_ids' in f['metadata']
                check(f"{name}: pair_ids present (R-18)", has_pairs, "No pair_ids in metadata")

                # Check edge types (R-17) on first 5 graphs
                bad_edges = False
                for idx in range(min(5, n_samples)):
                    g = f['graphs'][str(idx)]
                    ea_key = 'edge_type' if 'edge_type' in g else 'edge_attr'
                    if ea_key in g:
                        ea = g[ea_key][:]
                        if len(ea) > 0 and ea.max() > 3:
                            bad_edges = True
                check(f"{name}: edge types in {{0,1,2,3}} (R-17)", not bad_edges,
                      "Edge type >= 4 found (CDG leak?)")

                print(f"    {name}: {n_samples} samples")

# ---- R-19: Code lookup JSONL loads ----
check("R-19: samples.jsonl exists", os.path.exists(CODE_JSONL), f"{CODE_JSONL} not found")
if os.path.exists(CODE_JSONL):
    n_codes = 0
    with open(CODE_JSONL, encoding='utf-8') as f:
        for line in f:
            n_codes += 1
            if n_codes >= 10:  # quick check
                break
    check("R-19: JSONL has records", n_codes > 0, "Empty JSONL")

# ---- R-05/R-06: Config invariants ----
from training.scripts.model.run_ablations import (
    BASE_CONFIG, ABLATION_CONFIGS, assert_ablation_invariants
)
try:
    assert_ablation_invariants()
    check("R-05/R-06: All configs same seed + data paths", True)
except AssertionError as e:
    check("R-05/R-06: Config invariants", False, str(e))

# ---- R-03: GroupNorm check ----
from training.scripts.model.model import StreamGuardModel
test_model = StreamGuardModel(node_feature_dim=824, use_graph=True)
all_groupnorm = all(isinstance(n, nn.GroupNorm) for n in test_model.ggnn_norm)
check("R-03: GGNN uses GroupNorm (not BatchNorm)", all_groupnorm,
      "BatchNorm found -- will break at serving time")

# ---- R-15: Config B has single_conv ----
test_model_b = StreamGuardModel(node_feature_dim=824, type_aware_edges=False)
has_single_conv = hasattr(test_model_b, 'single_conv') and len(test_model_b.single_conv) == 3
check("R-15: Config B has dedicated single_conv", has_single_conv,
      "type-blind model missing single_conv")
del test_model, test_model_b
torch.cuda.empty_cache()

print(f"\n{'=' * 60}")
print(f"RESULT: {checks_passed}/{checks_total} checks passed")
if checks_passed < checks_total:
    print("FIX ALL FAILURES BEFORE PROCEEDING!")
else:
    print("All checks passed. Safe to proceed with training.")
print(f"{'=' * 60}")

## Cell 6: Training Configuration

Select which configs to run and training mode.

**Options:**
- `DRY_RUN = True`: 2 epochs, 1 batch per config (smoke test, ~5 min total)
- `DRY_RUN = False`: Full training, 20 epochs with early stopping (~3-5 hours per config)
- `CONFIGS_TO_RUN`: list of config names, or `None` for all 6

In [ ]:
# ======================================================================
# TRAINING CONFIGURATION -- EDIT THIS CELL
# ======================================================================

# Set True for smoke test (2 epochs, 1 batch). Set False for full training.
DRY_RUN = True

# Which configs to run. Set to None for all 6.
# Options: 'A_baseline', 'B_plus_ggnn', 'B_prime_type_aware',
#          'C_plus_cfa', 'D_plus_tpg', 'E_full'
CONFIGS_TO_RUN = None  # None = all 6 configs
# CONFIGS_TO_RUN = ['A_baseline', 'B_plus_ggnn']  # example: just 2 configs
# CONFIGS_TO_RUN = ['E_full']  # example: just the full model

# VRAM-based batch size (auto-detected from Cell 1)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_mem / 1024**3
if gpu_mem_gb >= 40:    # A100
    BATCH_SIZE = 16
    GRAD_ACCUM = 2
elif gpu_mem_gb >= 14:  # T4/V100
    BATCH_SIZE = 8
    GRAD_ACCUM = 4
else:                   # Small GPU
    BATCH_SIZE = 4
    GRAD_ACCUM = 8

# Override paths to use local SSD copies
DATA_OVERRIDES = {
    'train_h5': TRAIN_H5,
    'val_h5': VAL_H5,
    'test_h5': TEST_H5,
    'checkpoint_dir': CHECKPOINT_DIR,
    'batch_size_graphs': BATCH_SIZE,
    'gradient_accumulation': GRAD_ACCUM,
}

print(f"Mode: {'DRY RUN (smoke test)' if DRY_RUN else 'FULL TRAINING'}")
print(f"Configs: {CONFIGS_TO_RUN or 'ALL 6'}")
print(f"Batch size: {BATCH_SIZE}, Grad accum: {GRAD_ACCUM} (effective batch: {BATCH_SIZE * GRAD_ACCUM})")
print(f"GPU: {gpu_name} ({gpu_mem_gb:.0f} GB)")

## Cell 7: Run Ablation Training

This is the main training cell. Each config trains independently with:
- Same seed=42 (R-06)
- Same data files (R-05)
- Checkpoints saved every epoch (R-08)
- AMP enabled for VRAM efficiency (R-09, R-14)
- Gradient clipping at 1.0 (R-10)
- Test.h5 evaluated ONCE at end per config (R-01)

**Runtime estimates:**
| GPU | Per Config | All 6 Configs |
|-----|-----------|---------------|
| T4 (16 GB) | ~4-5 hours | ~24-30 hours |
| V100 (32 GB) | ~2-3 hours | ~12-18 hours |
| A100 (40 GB) | ~1.5-2 hours | ~9-12 hours |

In [ ]:
import time
from training.scripts.model.run_ablations import (
    ABLATION_CONFIGS, CONFIG_ORDER, run_ablations, print_ablation_table,
)
from training.scripts.model.train import train

# Apply data path overrides to all configs
for config_name in ABLATION_CONFIGS:
    ABLATION_CONFIGS[config_name].update(DATA_OVERRIDES)

configs = CONFIGS_TO_RUN or list(CONFIG_ORDER)

print(f"Starting ablation study: {len(configs)} configs")
print(f"Mode: {'DRY RUN' if DRY_RUN else 'FULL TRAINING'}")
print(f"Configs: {configs}")
print()

start_time = time.time()

results = run_ablations(
    configs_to_run=configs,
    dry_run=DRY_RUN,
)

total_time = time.time() - start_time
print(f"\nTotal training time: {total_time / 3600:.1f} hours")

## Cell 8: Ablation Results Table (Paper Table 2)

In [ ]:
# Print the formatted ablation table
print_ablation_table(results)

# Also print per-CWE F1 for configs that have it
for config_name in CONFIG_ORDER:
    if config_name not in results:
        continue
    r = results[config_name]
    per_cwe = r.get('per_cwe_f1', {})
    if per_cwe:
        print(f"\nPer-CWE F1 for {config_name}:")
        for cwe, f1_val in sorted(per_cwe.items()):
            target = 0.88
            status = 'PASS' if f1_val >= target else 'MISS'
            print(f"  {cwe:<12}: {f1_val:.4f} [{status}]")

## Cell 9: Save Checkpoints + Results to Drive

Copy all checkpoints and the ablation table JSON to Google Drive for persistence.

In [ ]:
import json
import shutil

# Create Drive output directories
drive_ckpt_dir = os.path.join(DRIVE_FOLDER, 'checkpoints')
drive_results_dir = os.path.join(DRIVE_FOLDER, 'results')
os.makedirs(drive_ckpt_dir, exist_ok=True)
os.makedirs(drive_results_dir, exist_ok=True)

# Copy checkpoints
if os.path.exists(CHECKPOINT_DIR):
    ckpt_files = [f for f in os.listdir(CHECKPOINT_DIR) if f.endswith('.pt')]
    print(f"Copying {len(ckpt_files)} checkpoints to Drive...")
    for f in ckpt_files:
        src = os.path.join(CHECKPOINT_DIR, f)
        dst = os.path.join(drive_ckpt_dir, f)
        shutil.copy2(src, dst)
        size_mb = os.path.getsize(src) / 1024**2
        print(f"  {f}: {size_mb:.0f} MB")

# Copy ablation table JSON
local_table = os.path.join(RESULTS_DIR, 'ablation_table.json')
if os.path.exists(local_table):
    shutil.copy2(local_table, os.path.join(drive_results_dir, 'ablation_table.json'))
    print(f"  ablation_table.json copied")

# Also save results directly from this run
results_path = os.path.join(drive_results_dir, 'ablation_results_latest.json')
with open(results_path, 'w') as f:
    # Convert any non-serializable types
    import numpy as np
    def default_serializer(obj):
        if isinstance(obj, (np.floating, np.integer)):
            return float(obj)
        raise TypeError(f"Not serializable: {type(obj)}")
    json.dump(results, f, indent=2, default=default_serializer)
print(f"  Results saved to {results_path}")

# Copy MLflow runs if they exist
mlruns_dir = os.path.join(WORK_DIR, 'mlruns')
if os.path.exists(mlruns_dir):
    drive_mlruns = os.path.join(DRIVE_FOLDER, 'mlruns')
    if not os.path.exists(drive_mlruns):
        print(f"  Copying MLflow runs to Drive...")
        shutil.copytree(mlruns_dir, drive_mlruns)
    else:
        # Incremental copy
        shutil.copytree(mlruns_dir, drive_mlruns, dirs_exist_ok=True)
    print(f"  MLflow runs saved")

print(f"\nAll outputs saved to: {DRIVE_FOLDER}")

## Cell 10: Training Summary

Final summary of all training results and GPU usage.

In [ ]:
print("=" * 70)
print("PHASE 4 TRAINING SUMMARY")
print("=" * 70)

# GPU info
if torch.cuda.is_available():
    peak_mem = torch.cuda.max_memory_allocated() / 1024**3
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Peak GPU memory: {peak_mem:.2f} GB")
    print(f"Total training time: {total_time / 3600:.1f} hours")
    print()

# Per-config summary
print(f"{'Config':<24} {'Val F1':>8} {'Test F1':>8} {'Status':>10}")
print("-" * 55)
for config_name in CONFIG_ORDER:
    if config_name not in results:
        continue
    r = results[config_name]
    if 'error' in r:
        print(f"{config_name:<24} {'--':>8} {'--':>8} {'FAILED':>10}")
    else:
        val_f1 = r.get('best_val_f1', 0)
        test_f1 = r.get('test_f1', 0)
        status = 'OK' if test_f1 > 0 else 'NO TEST'
        print(f"{config_name:<24} {val_f1:>8.4f} {test_f1:>8.4f} {status:>10}")
print("-" * 55)

# Key paper claims
print("\nKey Paper Claims:")
if 'C_plus_cfa' in results and 'B_prime_type_aware' in results:
    c_f1 = results['C_plus_cfa'].get('test_f1', 0)
    b_f1 = results['B_prime_type_aware'].get('test_f1', 0)
    delta = c_f1 - b_f1
    verdict = 'CONFIRMED' if delta > 0 else 'NOT CONFIRMED'
    print(f"  N1 (CFA works):  C({c_f1:.4f}) - B'({b_f1:.4f}) = {delta:+.4f}  [{verdict}]")

if 'D_plus_tpg' in results and 'C_plus_cfa' in results:
    d_f1 = results['D_plus_tpg'].get('test_f1', 0)
    c_f1 = results['C_plus_cfa'].get('test_f1', 0)
    delta = d_f1 - c_f1
    verdict = 'CONFIRMED' if delta > 0 else 'NOT CONFIRMED'
    print(f"  N3 (TPG works):  D({d_f1:.4f}) - C({c_f1:.4f}) = {delta:+.4f}  [{verdict}]")

if 'E_full' in results and 'D_plus_tpg' in results:
    e_f1 = results['E_full'].get('test_f1', 0)
    d_f1 = results['D_plus_tpg'].get('test_f1', 0)
    delta = e_f1 - d_f1
    verdict = 'CONFIRMED' if delta > 0 else 'NOT CONFIRMED'
    print(f"  N5 (Interproc):  E({e_f1:.4f}) - D({d_f1:.4f}) = {delta:+.4f}  [{verdict}]")

print(f"\n{'=' * 70}")
print("Next: Download results from Drive and prepare paper Table 2.")
print(f"{'=' * 70}")